<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-10/NB10_neural_networks.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 10: Neural Networks: From the Perceptron to Deep Learning

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook trains a perceptron on separable data and shows its failure on XOR, then writes a two-layer network with hand-derived backpropagation in NumPy and verifies the gradients numerically. In PyTorch it trains a network for the net electrical output of a combined cycle power plant from ambient temperature, exhaust vacuum, pressure and humidity [1], with standardisation, mini-batches, Adam, early stopping and a fair comparison with linear regression and a random forest [2, 3].

This notebook is the hands-on part of Week 10. It opens with Python step 10: Matrices, shapes and tensors and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-10/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-10/Week10_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-10/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-10/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 10 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

Colab provides PyTorch. A GPU is not needed this week; the code uses one automatically if it is available.

The last lines of the setup cell add a hint to every `NameError`. This error appears when a cell uses a name that no cell has defined in the current session, for example because the cells above have not run.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "on", DEVICE)

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

# Hint for a NameError: The usual cause in a notebook is a cell above that has not run in this session.
from IPython import get_ipython


def name_error_hint(shell, etype, value, tb, tb_offset=None):
    """Show a NameError as usual and add the hint."""
    shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("Hint: A name defined in a cell above exists only after that cell has run in this session. "
          "Runtime > Run before (Ctrl+F8) runs all cells above the current one.")


if get_ipython() is not None:
    get_ipython().set_custom_exc((NameError,), name_error_hint)

## Python step 10: Matrices, shapes and tensors

> **Starting here?** Run the setup cell above first. The first code cell below checks this and stops with a message if the setup has not run in this session.

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Vectors, matrices and the @ operator

A layer of a neural network multiplies its inputs by a weight matrix and adds a bias. When a batch of inputs is stored as a matrix X with one example per row, the whole layer is `X @ W + b`, where `@` is the matrix product. The shapes must match: A matrix of shape (4, 2) times one of shape (2, 2) gives shape (4, 2). NumPy adds the bias vector to every row automatically, a rule called broadcasting. The network below solves XOR with hand-set weights: One hidden unit acts as OR, the other as AND, and the output fires for OR but not for AND.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell above has not run in this session. "
                       "Run it first, then continue with this step.")

In [ ]:
import numpy as np

X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])   # four input examples
W1 = np.array([[1, 1], [1, 1]])                   # column 0: OR unit, column 1: AND unit
b1 = np.array([-0.5, -1.5])
H = (X @ W1 + b1 > 0).astype(int)                 # step activation
w2, b2 = np.array([1, -1]), -0.5
y = (H @ w2 + b2 > 0).astype(int)
print(X.shape, W1.shape, H.shape)
print(H)
print(y)

### Activation functions

Smooth activation functions replace the step so that gradients exist. The sigmoid squashes any number into the range 0 to 1, the hyperbolic tangent into -1 to 1, and the rectified linear unit keeps positive values and sets negative ones to zero. Written with NumPy, each of them works on whole arrays.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(z, 0)


z = np.array([-2.0, 0.0, 2.0])
print(sigmoid(z).round(3), np.tanh(z).round(3), relu(z))

### Tensors

PyTorch stores data in tensors, which behave much like NumPy arrays but can also run on graphics processors and record the operations applied to them [2]. Neural network code usually works with 32-bit floats, so conversions from NumPy name the type explicitly. `.shape` works as in NumPy, `.item()` extracts a single number, and `.numpy()` converts back to an array.

In [ ]:
import torch

Xt = torch.tensor(X, dtype=torch.float32)
W1t = torch.tensor(W1, dtype=torch.float32)
print(Xt.shape, Xt.dtype)
print((Xt @ W1t).sum().item(), (Xt @ W1t).numpy().tolist())

### Gradients with automatic differentiation

Training needs the derivative of the loss with respect to every weight. When a tensor is created with `requires_grad=True`, PyTorch records every operation on it, and `backward` computes the derivatives with the chain rule. For the loss L = (w x - y)^2 with w = 0.5, x = 2 and y = 3, the derivative is 2 (w x - y) x = -8. A gradient descent step then moves w against the gradient, and `torch.no_grad()` stops the recording while the weight is updated.

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
x_in, target = torch.tensor(2.0), torch.tensor(3.0)
loss = (w * x_in - target) ** 2
loss.backward()
print(loss.item(), w.grad.item())
with torch.no_grad():
    w -= 0.1 * w.grad                  # one gradient descent step with learning rate 0.1
print(round(w.item(), 4), round(((w * x_in - target) ** 2).item(), 4))

The loss falls from 4.0 to about 0.16 in one step. The training loops of this week repeat exactly this pattern for thousands of weights at once.

**Quick check.** A has shape (32, 10) and B has shape (10, 4). What is the shape of A @ B?

A. (32, 10)  
B. (10, 4)  
C. (32, 4)  
D. The product is not defined

<details><summary>Show the answer</summary>

**C.** The inner dimensions must agree, and the result keeps the outer ones: 32 rows and 4 columns.

</details>

### Exercises for Python step 10

**Exercise 10.1.** Replace both step activations of the XOR network by `sigmoid` and compute the output for the input `[1, 0]`, that is sigmoid(sigmoid(x @ W1 + b1) @ w2 + b2). Store the number in `py_soft_out`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py10_soft_out_reference():
    import numpy as np
    s = lambda z: 1 / (1 + np.exp(-z))
    hdn = s(np.array([1, 0]) @ np.array([[1, 1], [1, 1]]) + np.array([-0.5, -1.5]))
    return float(s(hdn @ np.array([1, -1]) - 0.5))

In [ ]:
py_soft_out = None
check("Exercise 10.1", py_soft_out, _py10_soft_out_reference())

**Exercise 10.2.** Use automatic differentiation to compute the derivative of L = (w x - y)^2 at w = 2.0, x = 2.0 and y = 3.0, and store it in `py_grad`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py10_grad_reference():
    return 2 * (2.0 * 2.0 - 3.0) * 2.0

In [ ]:
py_grad = None
check("Exercise 10.2", py_grad, _py10_grad_reference())

**Exercise 10.3.** Store the shape of `X @ W1` as a tuple in `py_shape`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py10_shape_reference():
    return (4, 2)

In [ ]:
py_shape = None
check("Exercise 10.3", py_shape, _py10_shape_reference())

### A reusable loader for public datasets

**Python move: a reusable function with keyword arguments.** `load_uci` downloads a dataset from the UCI Machine Learning Repository with the `ucimlrepo` package [4] and returns the features as a pandas DataFrame and the target as a Series. The package is installed on first use. If the repository cannot be reached, the calling cell switches to a documented synthetic stand-in, so that every later cell still runs.

In [ ]:
import subprocess
import sys


def load_uci(uci_id, target=None, drop=(), onehot_targets=False):
    """Return (X, y) for a UCI dataset; target names a column of the targets or of the features."""
    try:
        from ucimlrepo import fetch_ucirepo
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "15", "ucimlrepo"], check=False)
        from ucimlrepo import fetch_ucirepo
    data = fetch_ucirepo(id=uci_id).data
    X = data.features.copy()
    Y = data.targets.copy() if data.targets is not None else pd.DataFrame(index=X.index)
    if onehot_targets:                                   # several 0/1 target columns, one per class
        y = Y.idxmax(axis=1)
    else:
        if target is None:
            target = Y.columns[0]
        y = X.pop(target) if target in X.columns else Y[target]
    X = X.drop(columns=[c for c in drop if c in X.columns])
    return X, y.astype(float) if y.dtype.kind in "iuf" else y

## 1. One neuron

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

### Your turn, exercise 1

A neuron with weights w = (0.8, -0.5), bias b = 0.1 and a sigmoid activation receives x = (1.2, 2.0). Compute its output and store it in `neuron_out`.

**Python move: the `@` operator.** For NumPy arrays, `w @ x` is the dot product.

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _neuron_out_reference():
    return float(1 / (1 + np.exp(-(np.array([0.8, -0.5]) @ np.array([1.2, 2.0]) + 0.1))))

In [ ]:
neuron_out = None   # your computation
check("Exercise 1", neuron_out, _neuron_out_reference(), tol=1e-9)

## 2. The perceptron and its limit

> **Starting here?** Run the setup cell at the top of the notebook first, or choose **Runtime > Run before** (Ctrl+F8). The first code cell below checks this and stops with a message if the setup has not run in this session.

The perceptron rule updates the weights only after a mistake [5]. On separable data it converges; on XOR it cannot [6].

In [ ]:
#@title Check that the setup cell has run
if "check" not in globals():
    raise RuntimeError("The setup cell at the top of the notebook has not run in this session. "
                       "Run it first, or choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def perceptron(X, y, epochs=50, eta=0.1):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        mistakes = 0
        for xi, yi in zip(X, y):
            y_hat = 1 if xi @ w + b >= 0 else 0
            if y_hat != yi:
                w += eta * (yi - y_hat) * xi; b += eta * (yi - y_hat); mistakes += 1
        if mistakes == 0:
            break
    return w, b, np.mean((X @ w + b >= 0).astype(int) == y)


rng = np.random.default_rng(1)
X_sep = rng.normal(size=(100, 2)); y_sep = (X_sep[:, 0] + 0.5 * X_sep[:, 1] > 0).astype(int)
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float); y_xor = np.array([0, 1, 1, 0])
print("separable data, accuracy:", perceptron(X_sep, y_sep)[2])
print("XOR, accuracy:", perceptron(X_xor, y_xor)[2])

## 3. A two-layer network with backpropagation in NumPy

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

The network maps 2 inputs to 4 tanh units and 1 sigmoid output. For the cross-entropy loss, the error signal at the output is simply p - y; the chain rule carries it back through the weights and the derivative of tanh, 1 - h^2 [7].

**Python move: shapes.** `X` has shape (4, 2), `W1` (2, 4), the hidden layer `H` (4, 4) and the output (4, 1). Printing `.shape` is the quickest way to find a bug.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "X_xor", "y_xor"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def init(seed=0, hidden=4):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, 1, (2, hidden)), "b1": np.zeros(hidden), "W2": r.normal(0, 1, (hidden, 1)), "b2": np.zeros(1)}


def forward(p, X):
    H = np.tanh(X @ p["W1"] + p["b1"])
    P = 1 / (1 + np.exp(-(H @ p["W2"] + p["b2"])))
    return H, P


def loss_and_grads(p, X, y):
    H, P = forward(p, X)
    y = y.reshape(-1, 1)
    loss = -np.mean(y * np.log(P + 1e-12) + (1 - y) * np.log(1 - P + 1e-12))
    dZ2 = (P - y) / len(X)                         # output error signal
    dH = dZ2 @ p["W2"].T * (1 - H ** 2)            # back through tanh
    grads = {"W2": H.T @ dZ2, "b2": dZ2.sum(0), "W1": X.T @ dH, "b1": dH.sum(0)}
    return loss, grads


plt.figure(figsize=(5.5, 2.8))
for seed in [0, 1]:                                  # two different initial weight sets
    params, losses = init(seed), []
    for step in range(3000):
        loss, g = loss_and_grads(params, X_xor, y_xor)
        for k in params:
            params[k] -= 1.0 * g[k]
        losses.append(loss)
    print(f"seed {seed}: final loss {losses[-1]:.4f}, predictions {forward(params, X_xor)[1].ravel().round(3)}")
    plt.semilogy(losses, label=f"seed {seed}")
plt.xlabel("step"); plt.ylabel("cross-entropy"); plt.legend(); plt.title("XOR with a hidden layer"); plt.show()

With seed 0, gradient descent settles in a poor local minimum in which two of the four points receive 0.5; with seed 1, the network solves XOR. Training a network is a nonconvex optimisation problem, so results depend on the initial weights, as the results of the metaheuristics in Weeks 5 and 6 depended on their seeds.

### Your turn, exercise 2: gradient checking

A numerical gradient approximates a derivative by a central difference, (L(w + h) - L(w - h)) / (2h). For the freshly initialised network `init(seed=0)`, compute the numerical derivative of the loss on `X_xor` with respect to `W1[0, 1]` with h = 1e-5, and store it in `num_grad`. Compare it with the analytic value `loss_and_grads(init(0), X_xor, y_xor)[1]["W1"][0, 1]`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _num_grad_reference():
    p_plus, p_minus = init(0), init(0)
    p_plus["W1"][0, 1] += 1e-5; p_minus["W1"][0, 1] -= 1e-5
    return (loss_and_grads(p_plus, X_xor, y_xor)[0] - loss_and_grads(p_minus, X_xor, y_xor)[0]) / 2e-5

In [ ]:
num_grad = None   # your computation
check("Exercise 2", num_grad, _num_grad_reference(), tol=1e-6)

## 4. A combined cycle power plant

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

Tüfekci's dataset has 9568 hourly averages of ambient temperature (AT), exhaust vacuum (V), ambient pressure (AP) and relative humidity (RH), with the net electrical energy output (PE) in MW [1]. The stand-in, used only when the download fails, generates correlated inputs and an output with a nonlinear dependence on temperature.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "load_uci"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
def ccpp_stand_in(n=9568, seed=294):
    rng = np.random.default_rng(seed)
    AT = rng.uniform(2, 37, n)
    V = np.clip(25 + 1.3 * AT + rng.normal(0, 6, n), 25.4, 81.6)
    AP = rng.normal(1013, 6, n)
    RH = np.clip(95 - 1.2 * AT + rng.normal(0, 12, n), 25, 100)
    PE = 454.6 - 1.98 * AT - 0.234 * V + 0.062 * AP - 0.158 * RH - 0.012 * (AT - 20) ** 2 + rng.normal(0, 3.5, n)
    return pd.DataFrame({"AT": AT, "V": V, "AP": AP, "RH": RH}), pd.Series(PE, name="PE")


try:
    Xp, yp = load_uci(294, target="PE")
    PLANT_SOURCE = "UCI 294, Combined Cycle Power Plant"
except Exception as error:
    print("Download failed, using the synthetic stand-in:", type(error).__name__)
    Xp, yp = ccpp_stand_in()
    PLANT_SOURCE = "synthetic stand-in for UCI 294"
print(PLANT_SOURCE, Xp.shape)
X_tr, X_te, y_tr, y_te = train_test_split(Xp.to_numpy(float), yp.to_numpy(float), test_size=0.2, random_state=0)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.2, random_state=0)
sx, sy = StandardScaler().fit(X_tr), StandardScaler().fit(y_tr.reshape(-1, 1))
to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=DEVICE)
Xt, Xv, Xs = (to_t(sx.transform(a)) for a in (X_tr, X_va, X_te))
yt, yv = (to_t(sy.transform(a.reshape(-1, 1))) for a in (y_tr, y_va))

## 5. The network in PyTorch

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

**Python move: a class derived from nn.Module.** The layers are created once in `__init__`; `forward` describes how an input flows through them.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "Xt", "Xv", "yt", "yv"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
class PlantNet(nn.Module):
    def __init__(self, width=64, dropout=0.0):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(4, width), nn.ReLU(), nn.Dropout(dropout),
                                    nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 1))

    def forward(self, x):
        return self.layers(x)


def train_net(net, epochs=150, lr=1e-3, weight_decay=0.0, batch=64, patience=20, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=batch, shuffle=True)
    best, best_state, wait, hist = np.inf, None, 0, {"train": [], "val": []}
    for epoch in range(epochs):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss = nn.functional.mse_loss(net(xb), yb)
            loss.backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            tr, va = nn.functional.mse_loss(net(Xt), yt).item(), nn.functional.mse_loss(net(Xv), yv).item()
        hist["train"].append(tr); hist["val"].append(va)
        if va < best - 1e-5:
            best, wait = va, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    net.load_state_dict(best_state)
    return net, hist


net, hist = train_net(PlantNet().to(DEVICE))
plt.figure(figsize=(6, 3.2)); plt.plot(hist["train"], label="training"); plt.plot(hist["val"], label="validation")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("MSE (standardised)"); plt.legend(); plt.title("Learning curves"); plt.show()

### Your turn, exercise 3

Count the trainable parameters of a `PlantNet(width=64)` with `sum(p.numel() for p in model.parameters() if p.requires_grad)` and store the number in `n_params`. Then verify the number by hand from the layer sizes.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _n_params_reference():
    return sum(p.numel() for p in PlantNet(width=64).parameters() if p.requires_grad)

In [ ]:
n_params = None   # your computation
check("Exercise 3", n_params, _n_params_reference())

## 6. A fair comparison on the same test set

> **Starting here?** This section uses results of the cells above it, so choose **Runtime > Run before** (Ctrl+F8) first. The first code cell below checks this and stops with a message if they have not run in this session.

In [ ]:
#@title Check that the cells above have run
if not all(name in globals() for name in ["check", "net", "sy", "X_te", "X_tr", "Xs", "y_te", "y_tr"]):
    raise RuntimeError("This section uses results of the cells above, which have not run in this session. "
                       "Choose Runtime > Run before (Ctrl+F8), then continue with this section.")

In [ ]:
with torch.no_grad():
    pred_net = sy.inverse_transform(net(Xs).cpu().numpy()).ravel()
lin = LinearRegression().fit(X_tr, y_tr)
rf = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
rows = []
for name, pred in [("linear regression", lin.predict(X_te)), ("random forest", rf.predict(X_te)), ("neural network", pred_net)]:
    rows.append({"model": name, "test MAE (MW)": mean_absolute_error(y_te, pred), "test RMSE (MW)": mean_squared_error(y_te, pred) ** 0.5})
pd.DataFrame(rows).set_index("model").round(3)

## 7. Experiments to try

Change the width, add dropout with `PlantNet(width=128, dropout=0.2)`, try `weight_decay=1e-4`, or lower the learning rate, and compare learning curves and test errors. Run every setting with at least three seeds before drawing a conclusion: The differences between networks on this dataset are often smaller than the differences between seeds.

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-10/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-10/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-10/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-10/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-10/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 11](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-11/README.md) starts with its overview.

## References

[1] Tüfekci, P. (2014). Prediction of full load electrical power output of a base load operated combined cycle power plant using machine learning methods. *International Journal of Electrical Power & Energy Systems*, *60*, 126-140. <https://doi.org/10.1016/j.ijepes.2014.02.027>

[2] Paszke, A., Gross, S., Massa, F., et al. (2019). PyTorch: An imperative style, high-performance deep learning library. In *Advances in Neural Information Processing Systems 32* (pp. 8024-8035). <https://arxiv.org/abs/1912.01703>

[3] Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. In *3rd International Conference on Learning Representations (ICLR 2015)*. <https://arxiv.org/abs/1412.6980>

[4] Kelly, M., Longjohn, R., & Nottingham, K. (2026). *The UCI Machine Learning Repository*. <https://archive.ics.uci.edu>

[5] Rosenblatt, F. (1958). The perceptron: A probabilistic model for information storage and organization in the brain. *Psychological Review*, *65*(6), 386-408. <https://doi.org/10.1037/h0042519>

[6] Minsky, M., & Papert, S. (1969). *Perceptrons: An Introduction to Computational Geometry*. MIT Press.

[7] Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors. *Nature*, *323*(6088), 533-536. <https://doi.org/10.1038/323533a0>